In [68]:
import pandas as pd
import numpy as np

In [69]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [70]:
Payments = pd.read_csv('data/Internal/installments_payments.csv')

In [71]:
Payments.shape

(13605401, 8)

In [72]:
Payments.dtypes

SK_ID_PREV                  int64
SK_ID_CURR                  int64
NUM_INSTALMENT_VERSION    float64
NUM_INSTALMENT_NUMBER       int64
DAYS_INSTALMENT           float64
DAYS_ENTRY_PAYMENT        float64
AMT_INSTALMENT            float64
AMT_PAYMENT               float64
dtype: object

In [73]:
Payments = Payments.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [74]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [75]:
Payments = base.merge(Payments, on='SK_ID_CURR', how='inner')

In [76]:
Payments['MONTHS_BALANCE'] = (Payments['DAYS_INSTALMENT'] / 30).astype('int')

In [77]:
Payments.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'], inplace=True)

In [78]:
#Payments = Payments[Payments['MONTHS_BALANCE'] >= - 60]

Impute missing values

In [79]:
Payments['DAYS_ENTRY_PAYMENT'] = Payments.groupby('SK_ID_PREV')['DAYS_ENTRY_PAYMENT'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) #.reset_index()
Payments['AMT_PAYMENT'] = Payments.groupby('SK_ID_PREV')['AMT_PAYMENT'].transform(
    lambda x: x.ffill(limit=6).bfill(limit=6)
) #.reset_index()

Check if `NUM_INSTALMENT_VERSION` keeps changing or not.

In [80]:
Payments['INSTALMENT_VERSION_CHANGE'] = Payments.groupby(['SK_ID_PREV'])['NUM_INSTALMENT_VERSION'].diff().fillna(0).astype('int')

Aditional attributes

In [81]:
Payments['LATE_PAYMENT_DAYS'] = Payments['DAYS_ENTRY_PAYMENT'] - Payments['DAYS_INSTALMENT']

Payments['PAYMENT_AMT_TO_INSTALLMENT_RATIO'] = np.where(Payments['AMT_INSTALMENT'] > 0 ,
                                         Payments['AMT_PAYMENT'] / Payments['AMT_INSTALMENT'] , 
                                         np.nan)
Payments['LATE_PAYMENT_FLAG'] = (Payments['LATE_PAYMENT_DAYS'] > 0).astype('int')

Monthly Data

In [82]:
sum_cols = [
    'AMT_PAYMENT',
    'AMT_INSTALMENT',
    'INSTALMENT_VERSION_CHANGE'
]

In [83]:
Payments_Mon_Sum = Payments.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[sum_cols].sum()

In [84]:
Payments_Mon_Sum.columns = ['PMNT_INSTLMT_' + ''.join(col).strip().upper() for col in Payments_Mon_Sum.columns.values]
Payments_Mon_Sum = Payments_Mon_Sum.reset_index()

In [85]:
Payments_Mon_Sum.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'PMNT_INSTLMT_AMT_PAYMENT',
       'PMNT_INSTLMT_AMT_INSTALMENT',
       'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE'],
      dtype='object')

In [86]:
max_cols = [
    'LATE_PAYMENT_DAYS',
    'PAYMENT_AMT_TO_INSTALLMENT_RATIO',
    'LATE_PAYMENT_FLAG'
]

In [87]:
Payments_Mon_Max = Payments.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[max_cols].max()

In [88]:
Payments_Mon_Max.columns = ['PMNT_INSTLMT_' + ''.join(col).strip().upper() for col in Payments_Mon_Max.columns.values]
Payments_Mon_Max = Payments_Mon_Max.reset_index()

In [89]:
Payments_Mon_Max.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'PMNT_INSTLMT_LATE_PAYMENT_DAYS',
       'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO',
       'PMNT_INSTLMT_LATE_PAYMENT_FLAG'],
      dtype='object')

In [90]:
Payments_Mon = Payments_Mon_Sum.merge(Payments_Mon_Max, on=['SK_ID_CURR','MONTHS_BALANCE'], how='left')

In [91]:
del Payments_Mon_Max, Payments_Mon_Sum

Other Aggregates at previous ID level

In [102]:
Payments_Agg_PrevID = Payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    PMNT_INSTLMT_VRSN_CHANGES_COUNT = ('NUM_INSTALMENT_VERSION', lambda x: x.nunique() ), #number of unique installment version changes per previous application
    PMNT_INSTLMT_DURATION_DAYS = ('DAYS_ENTRY_PAYMENT', lambda x: x.max() - x.min() + 1), # duration of installment payments in days
    PMNT_INSTLMT_AVG_INSTALLMENT_FREQ = ('DAYS_ENTRY_PAYMENT', lambda x: pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan), # average frequency of installment payments in days
    PMNT_INSTLMT_NUM_INSTALMENTS = ('NUM_INSTALMENT_NUMBER', 'count'), # total number of installments for the previous application
).reset_index()

In [103]:
Payments_Agg_PrevID_counts = Payments_Agg_PrevID.groupby('SK_ID_CURR').agg({
    'PMNT_INSTLMT_VRSN_CHANGES_COUNT': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_DURATION_DAYS': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ': ['mean', 'max', 'min'],    
    'PMNT_INSTLMT_NUM_INSTALMENTS': ['mean', 'max', 'min']
})

In [104]:
Payments_Agg_PrevID_counts.columns = ['_'.join(col).strip().upper() for col in Payments_Agg_PrevID_counts.columns.values]
Payments_Agg_PrevID_counts = Payments_Agg_PrevID_counts.reset_index()

In [105]:
Payments_Agg_PrevID_Last1Y = Payments[Payments['MONTHS_BALANCE']>=-12].groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y = ('NUM_INSTALMENT_VERSION', lambda x: x.nunique() ), #number of unique installment version changes per previous application
    PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y = ('DAYS_ENTRY_PAYMENT', lambda x: pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan), # average frequency of installment payments in days
).reset_index()

In [106]:
Payments_Agg_PrevID_Last1Y_counts = Payments_Agg_PrevID_Last1Y.groupby('SK_ID_CURR').agg({
    'PMNT_INSTLMT_VRSN_CHANGES_COUNT_L1Y': ['mean', 'max', 'min'],
    'PMNT_INSTLMT_AVG_INSTALLMENT_FREQ_L1Y': ['mean', 'max', 'min']
})

In [107]:
Payments_Agg_PrevID_Last1Y_counts.columns = ['_'.join(col).strip().upper() for col in Payments_Agg_PrevID_Last1Y_counts.columns.values]
Payments_Agg_PrevID_Last1Y_counts = Payments_Agg_PrevID_Last1Y_counts.reset_index()

Previous Loan Data

In [99]:
PrevLoans = pd.read_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\prev_loan_data_processed.parquet")
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [108]:
Payments_PrevLoans = Payments_Agg_PrevID.merge(PrevLoans, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
Payments_PrevLoans['AMT_ANNUITY_TO_CREDIT_RATIO'] = Payments_PrevLoans['AMT_ANNUITY'] / Payments_PrevLoans['AMT_CREDIT']

In [109]:
Payments_Agg_PrevLoans = Payments_PrevLoans.groupby('SK_ID_CURR').agg(
    PMNT_INSTLMT_TOTAL_PREV_LOAN = ('SK_ID_PREV', 'nunique'),   
    PMNT_INSTLMT_AVG_CREDIT_TO_APPL = ('CREDIT_TO_APPL', 'mean'),
    PMNT_INSTLMT_ANNUITY_TO_CREDIT_RATIO = ('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),
    PMNT_INSTLMT_CNT_HIGH_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='high')*1).sum()),
    PMNT_INSTLMT_CNT_MIDDLE_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='middle')*1).sum()),
    PMNT_INSTLMT_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN = ('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    PMNT_INSTLMT_AVG_DAYS_DECISION_PREVLOAN = ('DAYS_DECISION', 'mean')
)

In [110]:
del Payments_Agg_PrevID

`PMNT_AVG_INSTALLMENT_FREQ` the average days between payments, is negative in few cases. This is because in some IDs, the `DAYS_ENTRY_PAYMENT` is decreasng with Installment Number , ie higher installment number indicates an earlier payment rather than a later payment. `NUM_INSTALMENT_NUMBER` is not in correct order, need to sort data based on `DAYS_ENTRY_PAYMENT`.

**last 1 Year**

In [111]:
agg_cols = [
    'PMNT_INSTLMT_AMT_PAYMENT',
    'PMNT_INSTLMT_AMT_INSTALMENT', 
    'PMNT_INSTLMT_INSTALMENT_VERSION_CHANGE',
    'PMNT_INSTLMT_LATE_PAYMENT_DAYS', 
    'PMNT_INSTLMT_PAYMENT_AMT_TO_INSTALLMENT_RATIO',
    'PMNT_INSTLMT_LATE_PAYMENT_FLAG'
]

In [112]:
Payments_Mon_1Y_Agg = Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-12].groupby('SK_ID_CURR')[agg_cols].agg(['mean', 'max', 'min'])
Payments_Mon_1Y_Agg.columns = ['_'.join(col).strip().upper() + '_1Y' for col in Payments_Mon_1Y_Agg.columns.values]

Last Quarter

In [113]:
Payments_Mon_3M_Agg =  Payments_Mon[Payments_Mon['MONTHS_BALANCE']>=-3].groupby('SK_ID_CURR')[agg_cols].agg(['mean', 'max', 'min'])
Payments_Mon_3M_Agg.columns = ['_'.join(col).strip().upper() + '_3M' for col in Payments_Mon_3M_Agg.columns.values]

Last Snapshot

In [115]:
Payments_Last_Snapshot= Payments_Mon.groupby(['SK_ID_CURR'])[agg_cols + ['MONTHS_BALANCE']].last().reset_index()
Payments_Last_Snapshot.columns = [col + '_LAST_MONTH' if col != 'SK_ID_CURR' else col for col in Payments_Last_Snapshot.columns]
Payments_Last_Snapshot.rename(columns={'MONTHS_BALANCE':'PMNT_INSTALMENT_DAYS_SINCE_LAST_TRANSACTION'}, inplace=True)

Merge all

In [116]:
Payments_Agg_final = Payments_Agg_PrevID_counts.merge(Payments_Agg_PrevID_Last1Y_counts, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_1Y_Agg, on='SK_ID_CURR', how='left', indicator=True )
Payments_Agg_final['PMNT_INSTLMT_IN_LAST_1Y'] = Payments_Agg_final['_merge'].apply(lambda x: 1 if x == 'both' else 0)
Payments_Agg_final.drop(columns=['_merge'], inplace=True)

Payments_Agg_final = Payments_Agg_final.merge(Payments_Mon_3M_Agg, on='SK_ID_CURR', how='left', indicator=True)
Payments_Agg_final['PMNT_INSTLMT_IN_LAST_3M'] = Payments_Agg_final['_merge'].apply(lambda x: 1 if x == 'both' else 0)
Payments_Agg_final.drop(columns=['_merge'], inplace=True)

Payments_Agg_final = Payments_Agg_final.merge(Payments_Last_Snapshot, on='SK_ID_CURR', how='left')
Payments_Agg_final = Payments_Agg_final.merge(Payments_Agg_PrevLoans, on='SK_ID_CURR', how='left')

In [118]:
Payments_Agg_final = Payments_Agg_final.merge(base, on='SK_ID_CURR', how='inner')

In [119]:
Payments_Agg_final.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\payment_installments_processed.parquet")